# Finetune ANY LLM MODEL to Generate Would You Rather Questions
Fine-tunes any LLM model on a dataset of WYR questions so it can generate new ones in the same vibe.


In [1]:
import random
import json
from tqdm import tqdm

import os
from dataclasses import dataclass, field
from typing import Dict, List, Any

import json
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
import torch

In [2]:

# Check if CUDA is available
print(f"Is CUDA available? {torch.cuda.is_available()}")

# Get the name of the GPU
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


Is CUDA available? False


/home/milk/Desktop/RESEARCH/wyr/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [3]:
from huggingface_hub import login
login()

In [4]:
# Load model directly

# list of models: google/gemma-3-1b-it, HuggingFaceTB/SmolLM2-1.7B-Instruct, TinyLlama/TinyLlama-1.1B-Chat-v1.0, Qwen/Qwen2.5-1.5B


BASE_MODEL_ID = "meta-llama/Llama-3.2-1B"
MODEL_NAME = "mz-llama"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    torch_dtype="auto"
)

/home/milk/Desktop/RESEARCH/wyr/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [5]:
# set padding token
if MODEL_NAME != "gemma":
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id
else:
    tokenizer.eos_token_id = 106
    tokenizer.eos_token = tokenizer.convert_ids_to_tokens(106)
    model.config.eos_token_id = 106
    model.config.eos_token = tokenizer.eos_token

## Dataset
`formatted_polls.json` contains 313 WYR questions with `title`, `optionA`, and `optionB`.

Each example is formatted as a simple generation prompt so the model learns the style and structure of WYR questions:
```
Would you rather [optionA] or [optionB]?
```
At inference time, priming the model with `Would you rather` will generate new questions in the same vibe.


In [6]:
from datetime import datetime
import os

date = datetime.now().strftime("%Y-%m-%d")

POLLS_JSON    = "formatted_polls.json"   # <-- put your file here
JSONL_OUTPUT  = f"train-data/wyr_training_data-[{date}].jsonl"

os.makedirs("train-data", exist_ok=True)


In [7]:


def build_wyr_dataset(polls_path: str) -> Dataset:
    """
    Format each WYR poll as a plain generation string.
    The model learns to produce well-formed WYR questions by
    predicting the full text token-by-token.

    Two formats are included so the model learns both the
    full question and the two-option layout:

      Format A (plain from the title):
        Would you rather [optionA] or [optionB]?

      Format B (structured):
        Would you rather...
        A) [optionA]
        B) [optionB]
    """
    with open(polls_path, 'r') as f:
        polls = json.load(f)

    rows = []
    for poll in polls:
        ctx = poll['title'].strip().rstrip('.')
        a = poll['optionA'].strip().rstrip('.')
        b = poll['optionB'].strip().rstrip('.')

        # Randomly swap A/B so the model doesn't develop order bias
        if random.random() < 0.5:
            a, b = b, a
        
        # Format A — plain from the title
        rows.append({'text': f"{ctx}\nWould you rather...\nA) {a}\nB) {b}?"})

        # Format B — structured two-option layout
        rows.append({'text': f"Would you rather...\nA) {a}\nB) {b}"})

    random.shuffle(rows)

    # Save for inspection
    with open(JSONL_OUTPUT, 'w') as f:
        for row in rows:
            f.write(json.dumps(row) + '\n')

    print(f"Built {len(rows)} training examples ({len(polls)} polls x 2 formats) -> {JSONL_OUTPUT}")
    return Dataset.from_list(rows)

train_dataset = build_wyr_dataset(POLLS_JSON)
print(train_dataset)
print("\nSample entries:")
for i in range(4):
    print(f"\n--- {i} ---")
    print(train_dataset[i]['text'])


Built 626 training examples (313 polls x 2 formats) -> train-data/wyr_training_data-[2026-05-22].jsonl
Dataset({
    features: ['text'],
    num_rows: 626
})

Sample entries:

--- 0 ---
Would you rather...
A) Have only verseasoned food
B) Have no seasonings

--- 1 ---
Would you rather...
A) Travel to only 1 universe of your choice
B) Travel to any universe at any time, but you cant choose

--- 2 ---
AHHHHH !!! God has CURSED everyone with a speaking curse. Would you rather…
Would you rather...
A) Be the only person in the world who cant speak
B) Be the only person in the world who can speak?

--- 3 ---
Would you rather always have your room be the perfect temperature when going to sleep or be able to immediately find the perfect sleeping position
Would you rather...
A) be able to immediately find the perfect sleeping position
B) always have your room be the perfect temperature when going to sleep?


## Train the model


In [8]:
# os.environ["OMP_NUM_THREADS"] = "6"
# os.environ["OPENBLAS_NUM_THREADS"] = "6"
# os.environ["MKL_NUM_THREADS"] = "6"
# os.environ["VECLIB_NUM_THREADS"] = "6"  
# os.environ["NUMEXPR_NUM_THREADS"] = "6"

# torch.set_num_threads(6)
# torch.set_num_interop_threads(2)

In [9]:
# Output directories
LORA_OUTPUT_DIR   = f"../model_outputs/{MODEL_NAME}-wyr-lora"
MERGED_OUTPUT_DIR = f"../model_outputs/{MODEL_NAME}-wyr-merged"

MAX_SEQ_LEN = 128  # WYR questions are short


peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

sft_config = SFTConfig(
    output_dir=LORA_OUTPUT_DIR,
    num_train_epochs=10,          # small dataset — multiple passes help
    dataloader_num_workers=4,
    dataloader_pin_memory=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=True,
    bf16=(
        torch.cuda.is_available()
        and torch.cuda.get_device_capability(0)[0] >= 8
    ),
    fp16=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

trainer.train()


[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported flash attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

Adding EOS to train dataset:   0%|          | 0/626 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/626 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/626 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.
/home/milk/Desktop/RESEARCH/wyr/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

In [ ]:
# 7) Save LoRA adapter + tokenizer locally
os.makedirs(LORA_OUTPUT_DIR, exist_ok=True)
trainer.model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"Saved LoRA adapter to {LORA_OUTPUT_DIR}")

# 8) Merge LoRA weights into base model and save a standalone model
print("Merging LoRA adapter into base model...")

# Reload base model on CPU (or cuda if you want)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="cpu",   # merge on CPU to avoid GPU OOM
)

lora_model = PeftModel.from_pretrained(base_model, LORA_OUTPUT_DIR)
merged_model = lora_model.merge_and_unload()  # apply LoRA weights into base

os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)
merged_model.save_pretrained(MERGED_OUTPUT_DIR)
tokenizer.save_pretrained(MERGED_OUTPUT_DIR)
print(f"Saved merged full model to {MERGED_OUTPUT_DIR}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Saved LoRA adapter to ../model_outputs/qwen-wyr-lora
Merging LoRA adapter into base model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved merged full model to ../model_outputs/qwen-wyr-merged


## Generate new Would You Rather questions


In [ ]:
# Quick generation test — prime the model with the WYR opener
# and let it complete the question.

model.eval()

prompts = [
    "Would you rather...",
    #"Would you rather...",
]
prompts *= 10

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"PROMPT: {prompt!r}")
    print(f"OUTPUT: {generated}")
    print()


PROMPT: 'Would you rather...'
OUTPUT: Would you rather... A) be a horse in ancient times B) Be an elephant during the romans?

PROMPT: 'Would you rather...'
OUTPUT: Would you rather... A) Never eat chocolate but 10% chance of severe hunger every hour B) Always have the perfect prefect burger on demand, otherwise an itchy finger that says “burger” everytime u touch urnt

PROMPT: 'Would you rather...'
OUTPUT: Would you rather... A) 120 years in a cell with no windows B) Be chased by evil rats for every wrong step u take?

PROMPT: 'Would you rather...'
OUTPUT: Would you rather... A) be the best dancer in your school (everyone thinks youre amazing but only one person knows B)
B) Be a really good artist, and everyone loves ur art BUT its hideous?

PROMPT: 'Would you rather...'
OUTPUT: Would you rather... A) never be able to touch the color green B) have your arms cut off?

PROMPT: 'Would you rather...'
OUTPUT: Would you rather... A) Have to live in california (current location)
B) Never be 

: 